# Bookstore Sales Analysis
Goal: analyze 200 synthetic sales of an online bookstore to find the top categories, the monthly trend, and unusual transactions.

## 1. Create the dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(5)
prices = {"Fiction": 14.99, "Non-Fiction": 18.99, "Children": 9.99,
          "Science": 24.99, "History": 21.99, "Comics": 12.99}
titles = {"Fiction": ["The Silent River", "Midnight Garden"],
          "Non-Fiction": ["Small Habits", "The Money Mind"],
          "Children": ["Leo the Brave Lion", "Stars for Mia"],
          "Science": ["Tiny Atoms", "The Living Planet"],
          "History": ["Roads of Rome", "The Long War"],
          "Comics": ["Captain Comet", "Night Patrol"]}

days = pd.date_range("2025-01-01", "2025-12-31")
weights = np.where(days.month == 12, 3, np.where(days.month == 11, 2, 1))
sales = pd.DataFrame({"date": rng.choice(days, 200, p=weights / weights.sum())})
sales["book category"] = rng.choice(list(prices), 200, p=[0.3, 0.2, 0.2, 0.1, 0.1, 0.1])
sales["title"] = [rng.choice(titles[c]) for c in sales["book category"]]
sales["units sold"] = rng.integers(1, 4, 200)
sales.loc[[40, 120, 175], "units sold"] = [20, 12, 15]
sales["unit price"] = (sales["book category"].map(prices) + rng.normal(0, 1.5, 200)).round(2)
sales["total revenue"] = (sales["units sold"] * sales["unit price"]).round(2)

sales = sales.sort_values("date")
sales.to_csv("sales_data.csv", index=False)
print("Saved sales_data.csv:", sales.shape)

## 2. Load and inspect the data

In [ ]:
df = pd.read_csv("sales_data.csv")
df.head()

In [ ]:
df.info()
print("Shape:", df.shape)

## 3. Descriptive statistics

In [ ]:
df[["total revenue", "units sold"]].agg(["mean", "median", "std", "min", "max"]).round(2)

## 4. Sales by category and by month

In [ ]:
by_category = df.groupby("book category").agg(
    transactions=("title", "count"),
    units=("units sold", "sum"),
    revenue=("total revenue", "sum"),
).sort_values("revenue", ascending=False).round(2)
by_category

In [ ]:
df["month"] = pd.to_datetime(df["date"]).dt.to_period("M")
by_month = df.groupby("month").agg(
    transactions=("title", "count"),
    revenue=("total revenue", "sum"),
).round(2)
by_month

## 5. Charts

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(by_category.index, by_category["revenue"], color="steelblue")
plt.title("Revenue by Category")
plt.xlabel("Book category")
plt.ylabel("Total revenue ($)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["total revenue"], bins=20, color="darkorange", edgecolor="black")
plt.title("Distribution of Transaction Amounts")
plt.xlabel("Total revenue per transaction ($)")
plt.ylabel("Number of transactions")
plt.show()

## 6. Outliers in total revenue (IQR method)

In [ ]:
q1 = df["total revenue"].quantile(0.25)
q3 = df["total revenue"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(f"Q1: ${q1:.2f} | Q3: ${q3:.2f} | IQR: ${iqr:.2f}")
print(f"Normal range: ${lower:.2f} to ${upper:.2f}")

outliers = df[(df["total revenue"] < lower) | (df["total revenue"] > upper)]
print("Outliers found:", len(outliers))
outliers

## 7. Conclusions
- **Top category:** Fiction, with $2,217.60 in revenue from 68 transactions. Non-Fiction is second, with $1,705.65.
- **Best month:** December 2025, with $1,772.37 from 43 transactions. November is second, with $898.34, so the holiday season drives sales.
- **Outliers:** the IQR method (normal range up to $82.51) found 3 bulk orders: 15 copies of Small Habits ($294.15), 20 of Captain Comet ($273.40), and 12 of The Silent River ($161.04). They are real sales, so I keep them.